# ReAct 框架 - 第一部分：基础概念

## 学习目标
1. 理解 ReAct 的核心原理
2. 掌握 Thought-Action-Observation 循环
3. 学习基础数据结构

## 目录
1. [什么是 ReAct](#1-什么是-react)
2. [ReAct vs CoT](#2-react-vs-cot)
3. [核心数据结构](#3-核心数据结构)
4. [Thought-Action-Observation](#4-thought-action-observation)
5. [练习](#5-练习)

In [ ]:
import sys
sys.path.insert(0, '..')

from src.react import (
    Thought, Action, Observation, ReActStep, ReActTrace,
    SimpleTool, ReActPromptBuilder, ReActParser, ReActAgent
)
print("模块加载成功！")

---
## 1. 什么是 ReAct

### 1.1 核心思想

ReAct (Reasoning + Acting) 结合推理和行动，让 LLM 能够与外部环境交互。

In [ ]:
print("""
ReAct 核心循环：

┌─────────────────────────────────────────────────────┐
│                                                     │
│    ┌──────────┐                                     │
│    │ Thought  │ ← 思考：分析当前状态，决定下一步     │
│    └────┬─────┘                                     │
│         │                                           │
│         ▼                                           │
│    ┌──────────┐                                     │
│    │  Action  │ ← 行动：调用工具或执行操作           │
│    └────┬─────┘                                     │
│         │                                           │
│         ▼                                           │
│    ┌──────────┐                                     │
│    │Observation│ ← 观察：获取行动结果               │
│    └────┬─────┘                                     │
│         │                                           │
│         └──────────────────┐                        │
│                            │                        │
│    重复直到任务完成 ◄───────┘                        │
│                                                     │
└─────────────────────────────────────────────────────┘
""")

### 1.2 为什么需要 ReAct

**纯推理的局限**：
- 知识可能过时
- 无法获取实时信息
- 无法执行实际操作

**ReAct 的优势**：
- 可以调用工具获取信息
- 可以执行实际操作
- 推理过程透明可追溯

In [ ]:
# 示例：查询天气
print("""
示例任务：查询北京今天的天气

纯 CoT 方式：
  Thought: 北京今天的天气...
  Answer: 我不知道，因为我没有实时数据

ReAct 方式：
  Thought: 我需要查询北京的天气，应该使用天气API
  Action: weather_api(city="北京")
  Observation: 晴，25°C，湿度60%
  Thought: 我已经获取了天气信息
  Answer: 北京今天晴，气温25°C，湿度60%
""")

---
## 2. ReAct vs CoT

### 2.1 对比分析

In [ ]:
print("""
┌──────────────┬─────────────────┬─────────────────┐
│     特性     │      CoT        │     ReAct       │
├──────────────┼─────────────────┼─────────────────┤
│ 核心能力     │ 推理            │ 推理 + 行动     │
│ 外部交互     │ 无              │ 可调用工具      │
│ 信息来源     │ 模型知识        │ 模型 + 外部     │
│ 适用任务     │ 推理、计算      │ 信息检索、操作  │
│ 复杂度       │ 低              │ 中              │
│ 可控性       │ 低              │ 高              │
└──────────────┴─────────────────┴─────────────────┘
""")

### 2.2 选择指南

In [ ]:
print("""
何时使用 CoT：
  ✓ 数学计算
  ✓ 逻辑推理
  ✓ 文本分析
  ✓ 不需要外部信息

何时使用 ReAct：
  ✓ 需要实时信息（天气、股价）
  ✓ 需要搜索（知识库、网页）
  ✓ 需要执行操作（发邮件、写文件）
  ✓ 多步骤任务
""")

---
## 3. 核心数据结构

### 3.1 Thought - 思考

In [ ]:
# 创建思考
thought1 = Thought(content="我需要查询北京的天气信息")
thought2 = Thought(content="已获取天气数据，现在可以回答用户")

print("Thought 示例：")
print(f"  思考1: {thought1.content}")
print(f"  思考2: {thought2.content}")

### 3.2 Action - 行动

In [ ]:
# 创建行动
action1 = Action(name="search", input="北京天气")
action2 = Action(name="calculator", input="15 * 24")

print("Action 示例：")
print(f"  行动1: 工具={action1.name}, 输入={action1.input}")
print(f"  行动2: 工具={action2.name}, 输入={action2.input}")

### 3.3 Observation - 观察

In [ ]:
# 创建观察
obs1 = Observation(content="北京今天晴，气温25°C")
obs2 = Observation(content="360")

print("Observation 示例：")
print(f"  观察1: {obs1.content}")
print(f"  观察2: {obs2.content}")

### 3.4 ReActStep - 完整步骤

In [ ]:
# 创建完整步骤
step = ReActStep(
    thought=thought1,
    action=action1,
    observation=obs1
)

print("ReActStep 完整步骤：")
print(f"  Thought: {step.thought.content}")
print(f"  Action: {step.action.name}({step.action.input})")
print(f"  Observation: {step.observation.content}")

### 3.5 ReActTrace - 执行轨迹

In [ ]:
# 创建执行轨迹
trace = ReActTrace(question="查询北京天气")
trace.add_step(step)

print(f"ReActTrace：")
print(f"  任务: {trace.question}")
print(f"  步骤数: {len(trace.steps)}")
print(f"  是否完成: {trace.success}")

---
## 4. Thought-Action-Observation

### 4.1 循环详解

In [ ]:
print("""
Thought-Action-Observation 循环详解：

1. Thought（思考）
   - 分析当前状态
   - 确定下一步行动
   - 解释推理过程

2. Action（行动）
   - 选择合适的工具
   - 构造工具输入
   - 执行工具调用

3. Observation（观察）
   - 接收工具返回结果
   - 解析结果内容
   - 为下一步思考提供信息

终止条件：
  - 任务完成
  - 达到最大步数
  - 遇到无法处理的错误
""")

### 4.2 完整示例

In [ ]:
# 模拟完整的 ReAct 循环
print("任务：计算 2024 年有多少小时")
print("="*50)

# 步骤1
print("\n步骤 1:")
print("  Thought: 2024年是闰年，有366天，我需要计算366天有多少小时")
print("  Action: calculator(366 * 24)")
print("  Observation: 8784")

# 步骤2
print("\n步骤 2:")
print("  Thought: 我已经计算出结果，可以回答了")
print("  Action: finish(2024年有8784小时)")
print("  Observation: 任务完成")

print("\n" + "="*50)
print("最终答案：2024年有8784小时")

---
## 5. 练习

### 练习1：创建 ReActStep

In [ ]:
# TODO: 为"搜索Python教程"任务创建一个 ReActStep
# my_thought = Thought(...)
# my_action = Action(...)
# my_observation = Observation(...)
# my_step = ReActStep(...)

### 练习2：创建 ReActTrace

In [ ]:
# TODO: 创建一个包含2个步骤的 ReActTrace
# my_trace = ReActTrace(question="...")
# my_trace.add_step(...)
# my_trace.add_step(...)

---
## 下一步

继续学习 **02b_ReAct_Tools.ipynb** 了解工具定义和使用